# NILM Disaggregation: Noise Robustness Evaluation

This notebook directly addresses the requirement to test the robustness of the multi-output model by injecting varying levels of artificial noise into the aggregate electrical signal.

**Goal:** Disaggregate the electrical signal under severe noise corruption (e.g. 0%, 5%, 15%, 30% noise) to see exactly when the model fails. As stated: *a positive result doesn’t matter - just do this to know where your model stands*.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.layers import *
from tensorflow.keras.models import Model
from sklearn.metrics import mean_absolute_error, r2_score
import json
import warnings
warnings.filterwarnings('ignore')

# 1. Load Preprocessed Data
df = pd.read_csv('iawe_preprocessed.csv')
APPLIANCES = ['AC', 'Motor']
FEATURE_COLS = ['Aggregate', 'Vrms', 'Irms', 'Reactive_Q', 'Apparent_Power', 'Power_Factor', 'Delta_P', 'Rolling_Mean', 'Rolling_Std', 'Hour']


In [ ]:
# 2. Scaling
app_max = {app: df[app].quantile(0.999) for app in APPLIANCES}
agg_max = df['Aggregate'].quantile(0.999)
feature_max = df[FEATURE_COLS].quantile(0.999)

df_scaled = df.copy()
df_scaled[FEATURE_COLS] = df_scaled[FEATURE_COLS] / feature_max

WINDOW = 99
STRIDE = 6
MIDPOINT = WINDOW // 2
N_FEATURES = len(FEATURE_COLS)

# Extract sequences
data_values = df_scaled[FEATURE_COLS].values
num_windows = (len(df_scaled) - WINDOW) // STRIDE + 1
X_all = np.zeros((num_windows, WINDOW, N_FEATURES))
Y_all = np.zeros((num_windows, len(APPLIANCES)))

for i in range(num_windows):
    s = i * STRIDE
    X_all[i] = data_values[s : s + WINDOW]
    for j, name in enumerate(APPLIANCES):
        Y_all[i, j] = df[name].values[s + MIDPOINT] / app_max[name]

# Train/Test Split (70/30)
np.random.seed(42)
indices = np.random.permutation(len(X_all))
split_idx = int(0.7 * len(X_all))
idx_train, idx_test = indices[:split_idx], indices[split_idx:]

X_train, X_test = X_all[idx_train], X_all[idx_test]
Y_train, Y_test = Y_all[idx_train], Y_all[idx_test]
print(f"Data Prepared: {len(X_test)} Test Windows")


In [ ]:
# 3. Model Architecture
def build_model():
    inp = Input(shape=(WINDOW, N_FEATURES), name='aggregate_input')
    x = GaussianNoise(0.005)(inp)
    
    x = Conv1D(30, 10, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Conv1D(30, 8, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Conv1D(40, 6, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Conv1D(50, 5, activation='relu')(x)
    x = Conv1D(50, 5, activation='relu')(x)

    x = Bidirectional(LSTM(64, return_sequences=True, dropout=0.2))(x)
    attn = Attention()([x, x])
    x = Add()([x, attn])
    x = LayerNormalization()(x)
    x = Flatten()(x)

    ac_head = Dense(512, activation='relu')(x)
    ac_head = Dropout(0.2)(ac_head)
    out_ac = Dense(1, activation='linear', name='out_ac')(ac_head) 

    motor_head = Dense(512, activation='relu')(x)
    motor_head = Dropout(0.2)(motor_head)
    out_motor = Dense(1, activation='linear', name='out_motor')(motor_head) 

    out = tf.keras.layers.Concatenate(name='multi_output')([out_ac, out_motor])
    model = Model(inputs=inp, outputs=out)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4), loss='mse', metrics=['mae'])
    return model

model = build_model()
# Fast training for evaluation purposes
print("Training model for robustness evaluation...")
model.fit(X_train, Y_train, validation_data=(X_test, Y_test), epochs=5, batch_size=128, verbose=1)


In [ ]:
# 4. Inject Noise and Evaluate
noise_levels = [0.0, 0.05, 0.15, 0.30] # 0%, 5%, 15%, 30% Noise
results = {app: [] for app in APPLIANCES}

for noise in noise_levels:
    print(f"\n--- Evaluating with {noise*100}% Noise ---")
    # Inject Noise (Assuming inputs are normalized around 0-1)
    noise_matrix = np.random.normal(0, noise, X_test.shape)
    X_test_noisy = X_test + noise_matrix
    
    raw_preds = model.predict(X_test_noisy, verbose=0)
    
    for j, app in enumerate(APPLIANCES):
        true_w = Y_test[:, j] * app_max[app]
        pred_w = np.clip(raw_preds[:, j] * app_max[app], 0, None)
        mae = mean_absolute_error(true_w, pred_w)
        results[app].append(mae)
        print(f"{app} MAE: {mae:.2f} W")


In [ ]:
# 5. Plot Robustness Degradation
plt.figure(figsize=(10, 6))
plt.plot([n*100 for n in noise_levels], results['AC'], marker='o', linewidth=2, color='#ff0000', label='AC MAE')
plt.plot([n*100 for n in noise_levels], results['Motor'], marker='s', linewidth=2, color='#ff9900', label='Motor MAE')

plt.title('Model Robustness: MAE vs Signal Noise Level', fontsize=15, fontweight='bold')
plt.xlabel('Artificial Gaussian Noise Level (%) injected into Test Set', fontsize=12)
plt.ylabel('Mean Absolute Error (Watts)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()

print("Conclusion: As requested, this shows exactly where the model stands when the signal degrades.")
